# 01 - Stratified Train/Val/Test Split

Loads the pipeline v0 feature table, separates features from metadata, and applies a stratified 70/15/15 split. Saves the splits (full rows, metadata included) to `data/splits/` as `.parquet`.

To use the definitive dataset, update `DATA_PATH` only (same column structure).

In [1]:
import sys
from pathlib import Path

_p = Path.cwd()
for ROOT in [_p, *_p.parents]:
    if (ROOT / "requirements.txt").exists():
        break
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from sklearn.model_selection import train_test_split

from preprocessing import load_feature_table, split_features_metadata

# === Configuration =========================================================
DATA_PATH = ROOT / "outputs" / "pilot" / "pipeline_v0" / "features_v0_ht40.csv"
SPLITS_DIR = ROOT / "data" / "splits"
RANDOM_SEED = 42
VAL_SIZE = 0.15
TEST_SIZE = 0.15

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_PATH:", DATA_PATH)

DATA_PATH: /home/xavier/code/wifi-csi-presence-detection/outputs/pilot/pipeline_v0/features_v0_ht40.csv


In [2]:
df = load_feature_table(DATA_PATH)
X, y, meta = split_features_metadata(df)
print("table:", df.shape, "| features:", X.shape[1],
      "| classes:", y.value_counts().to_dict())

table: (1200, 657) | features: 648 | classes: {0: 600, 1: 600}


In [3]:
# 70 / 15 / 15 stratified split in two stages, on full rows (metadata kept).
df_train, df_temp = train_test_split(
    df, test_size=VAL_SIZE + TEST_SIZE, stratify=df["label"], random_state=RANDOM_SEED
)
rel_test = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
df_val, df_test = train_test_split(
    df_temp, test_size=rel_test, stratify=df_temp["label"], random_state=RANDOM_SEED
)
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"{name}: {len(part)} rows | classes {part['label'].value_counts().to_dict()}")

train: 840 rows | classes {0: 420, 1: 420}
val: 180 rows | classes {0: 90, 1: 90}
test: 180 rows | classes {0: 90, 1: 90}


In [4]:
df_train.to_parquet(SPLITS_DIR / "train.parquet")
df_val.to_parquet(SPLITS_DIR / "val.parquet")
df_test.to_parquet(SPLITS_DIR / "test.parquet")
print("saved splits to", SPLITS_DIR)

saved splits to /home/xavier/code/wifi-csi-presence-detection/data/splits


> **Methodological note - window-level leakage.** The split is stratified by *window*, so correlated windows from the same session may appear in train, val, and test. This is acceptable for validating the pipeline with the pilot dataset (4 sessions). For the definitive dataset, consider `GroupKFold`/`GroupShuffleSplit` by `session_id` to avoid optimistic estimates.